In [5]:
import pandas as pd 

data=pd.read_csv('1_test.csv')
data.columns


Index(['class', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000',
       'ag_000', 'ag_001', 'ag_002',
       ...
       'ee_002', 'ee_003', 'ee_004', 'ee_005', 'ee_006', 'ee_007', 'ee_008',
       'ee_009', 'ef_000', 'eg_000'],
      dtype='object', length=171)

In [ ]:
import streamlit as st
from helper import opener


# ---------------------------------
# Page Configuration
# ---------------------------------

st.set_page_config(
    page_title="ScaniaGuard",
    page_icon="🚛",
    layout="wide"
)


# ---------------------------------
# Header
# ---------------------------------

st.title("🚛 ScaniaGuard")

st.subheader(
    "Cost-Sensitive APS Failure Detection System"
)

st.write(
    "Upload Scania truck operational data to "
    "detect potential APS failures."
)


# ---------------------------------
# Upload File
# ---------------------------------

uploaded_file = st.file_uploader(
    "Upload CSV file",
    type=["csv"]
)


# ---------------------------------
# Prediction
# ---------------------------------

if uploaded_file is not None:

    st.success("CSV uploaded successfully!")

    if st.button("Predict"):

        try:

            predictions, probabilities, y_true = opener(
                uploaded_file
            )

            # ---------------------------------
            # Prediction Summary
            # ---------------------------------

            total_trucks = len(predictions)

            predicted_failures = (
                predictions == 1
            ).sum()

            predicted_normal = (
                predictions == 0
            ).sum()

            st.subheader("Prediction Summary")

            col1, col2, col3 = st.columns(3)

            col1.metric(
                "Total Trucks",
                total_trucks
            )

            col2.metric(
                "APS Failures Detected",
                predicted_failures
            )

            col3.metric(
                "No APS Failure",
                predicted_normal
            )

            # ---------------------------------
            # Evaluation Mode
            # ---------------------------------

            if y_true is not None:

                st.subheader(
                    "📊 Evaluation Results"
                )

                tp = (
                    (y_true == 1) &
                    (predictions == 1)
                ).sum()

                tn = (
                    (y_true == 0) &
                    (predictions == 0)
                ).sum()

                fp = (
                    (y_true == 0) &
                    (predictions == 1)
                ).sum()

                fn = (
                    (y_true == 1) &
                    (predictions == 0)
                ).sum()

                # Cost
                total_cost = (
                    fp * 10
                    +
                    fn * 500
                )

                recall = (
                    tp / (tp + fn)
                    if (tp + fn) > 0
                    else 0
                )

                precision = (
                    tp / (tp + fp)
                    if (tp + fp) > 0
                    else 0
                )

                c1, c2, c3, c4 = st.columns(4)

                c1.metric(
                    "True Positives",
                    tp
                )

                c2.metric(
                    "False Positives",
                    fp
                )

                c3.metric(
                    "False Negatives",
                    fn
                )

                c4.metric(
                    "True Negatives",
                    tn
                )

                st.metric(
                    "💰 Total Cost",
                    f"₹{total_cost:,}"
                )

                c1, c2 = st.columns(2)

                c1.metric(
                    "Recall",
                    f"{recall:.2%}"
                )

                c2.metric(
                    "Precision",
                    f"{precision:.2%}"
                )

            # ---------------------------------
            # Results Table
            # ---------------------------------

            st.subheader("Prediction Results")

            results = {
                "Prediction": predictions,
                "Failure Probability": probabilities
            }

            result_df = __import__(
                "pandas"
            ).DataFrame(results)

            result_df["Prediction"] = result_df[
                "Prediction"
            ].map({
                0: "Not APS Failure",
                1: "APS Failure"
            })

            result_df["Failure Probability"] = (
                result_df["Failure Probability"]
                .map(lambda x: f"{x:.2%}")
            )

            st.dataframe(
                result_df,
                use_container_width=True
            )

        except Exception as e:

            st.error(
                f"Prediction failed: {e}"
            )

In [ ]:
import pandas as pd
import numpy as np
import joblib


# ---------------------------------
# Load Model Artifacts
# ---------------------------------

l_object = joblib.load("artifacts/model_things.joblib")

model = l_object["model"]
feature_columns = l_object["features"]
threshold = l_object["threshold"]
medians = l_object["medians"]
missing_columns = l_object["missing_names"]


# ---------------------------------
# Read CSV Automatically
# ---------------------------------

def read_scaniaguard_csv(input_file):

    input_file.seek(0)

    # Read the first few lines to identify the format
    first_lines = input_file.read(5000).decode(
        "utf-8",
        errors="ignore"
    )

    input_file.seek(0)

    # ---------------------------------
    # Case 1: Clean CSV
    # ---------------------------------

    if "aa_000" in first_lines:

        df = pd.read_csv(input_file)

        return df

    # ---------------------------------
    # Case 2: Original Scania CSV
    # ---------------------------------

    input_file.seek(0)

    df = pd.read_csv(
        input_file,
        skiprows=20
    )

    return df

# ---------------------------------
# Prediction Function
# ---------------------------------

def opener(input_file):

    # Read CSV
    df = read_scaniaguard_csv(input_file)

    # Convert "na" to NaN
    df = df.replace("na", np.nan)

    # ---------------------------------
    # Check whether labels exist
    # ---------------------------------

    has_labels = "class" in df.columns

    if has_labels:

        # Save actual labels
        y_true = df["class"].map({
            "pos": 1,
            "neg": 0
        })

        # Remove target column
        x = df.drop(columns=["class"])

    else:

        y_true = None
        x = df.copy()

    # ---------------------------------
    # Create Missingness Indicators
    # ---------------------------------

    for col in missing_columns:

        indicator_name = col + "_is_missing"

        x[indicator_name] = (
            x[col].isna().astype(int)
        )

    # ---------------------------------
    # Convert Features to Numeric
    # ---------------------------------

    x = x.apply(
        pd.to_numeric,
        errors="coerce"
    )

    # ---------------------------------
    # Median Imputation
    # ---------------------------------

    for name, value in medians.items():

        x[name] = x[name].fillna(value)

    # ---------------------------------
    # Check Required Features
    # ---------------------------------

    missing_features = list(
        set(feature_columns) - set(x.columns)
    )

    if missing_features:

        raise ValueError(
            f"Missing required features: "
            f"{missing_features[:10]}"
        )

    # ---------------------------------
    # Keep Exact Training Feature Order
    # ---------------------------------

    x = x[feature_columns]

    # ---------------------------------
    # Predict Probability
    # ---------------------------------

    probability = model.predict_proba(x)[:, 1]

    # ---------------------------------
    # Apply Cost-Sensitive Threshold
    # ---------------------------------

    prediction = (
        probability >= threshold
    ).astype(int)

    # ---------------------------------
    # Return Results
    # ---------------------------------

    return prediction, probability, y_true

### version 2


In [ ]:
import streamlit as st
import pandas as pd

from helper import opener


# =========================================================
# PAGE CONFIGURATION
# =========================================================

st.set_page_config(
    page_title="ScaniaGuard",
    page_icon="🚛",
    layout="wide",
    initial_sidebar_state="expanded"
)


# =========================================================
# CUSTOM CSS
# =========================================================

st.markdown(
    """
    <style>

    /* Main background */
    .stApp {
        background: linear-gradient(
            135deg,
            #0f172a 0%,
            #111827 50%,
            #172033 100%
        );
    }

    /* Main content width */
    .block-container {
        padding-top: 2rem;
        padding-bottom: 3rem;
        max-width: 1400px;
    }

    /* Header */
    .main-title {
        font-size: 3rem;
        font-weight: 800;
        margin-bottom: 0.2rem;
    }

    .subtitle {
        font-size: 1.15rem;
        color: #aab4c3;
        margin-bottom: 1.5rem;
    }

    /* Cards */
    .card {
        background: rgba(255, 255, 255, 0.06);
        border: 1px solid rgba(255, 255, 255, 0.10);
        border-radius: 18px;
        padding: 24px;
        margin-bottom: 20px;
        backdrop-filter: blur(10px);
    }

    .card-title {
        font-size: 1.25rem;
        font-weight: 700;
        margin-bottom: 10px;
    }

    .card-text {
        color: #b7c0cc;
        font-size: 0.95rem;
    }

    /* Status cards */
    .status-card {
        border-radius: 16px;
        padding: 20px;
        background: rgba(255, 255, 255, 0.06);
        border: 1px solid rgba(255, 255, 255, 0.10);
        text-align: center;
    }

    .status-number {
        font-size: 2rem;
        font-weight: 800;
    }

    .status-label {
        color: #aab4c3;
        font-size: 0.9rem;
    }

    /* Risk result */
    .risk-high {
        background: rgba(220, 38, 38, 0.15);
        border: 1px solid rgba(239, 68, 68, 0.4);
        border-radius: 18px;
        padding: 25px;
        text-align: center;
    }

    .risk-low {
        background: rgba(22, 163, 74, 0.15);
        border: 1px solid rgba(34, 197, 94, 0.4);
        border-radius: 18px;
        padding: 25px;
        text-align: center;
    }

    .risk-title {
        font-size: 1.7rem;
        font-weight: 800;
    }

    .risk-probability {
        font-size: 2.5rem;
        font-weight: 800;
        margin-top: 8px;
    }

    /* Buttons */
    .stButton > button {
        width: 100%;
        border-radius: 10px;
        height: 3rem;
        font-weight: 700;
        font-size: 1rem;
    }

    /* Sidebar */
    section[data-testid="stSidebar"] {
        background: #0b1220;
    }

    /* Divider */
    .custom-divider {
        height: 1px;
        background: rgba(255,255,255,0.1);
        margin: 25px 0;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# =========================================================
# HEADER
# =========================================================

st.markdown(
    '<div class="main-title">🚛 ScaniaGuard</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Cost-Sensitive APS Failure Detection System'
    '</div>',
    unsafe_allow_html=True
)


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.markdown("## 🚛 ScaniaGuard")

    st.markdown(
        """
        ### Model Information

        **Algorithm**  
        XGBoost

        **Features**  
        236

        **Decision Threshold**  
        0.024

        **False Positive Cost**  
        ₹10

        **False Negative Cost**  
        ₹500
        """
    )

    st.markdown("---")

    st.markdown(
        """
        ### How it works

        1. Upload truck data
        2. Preprocess missing values
        3. Generate missingness indicators
        4. Run XGBoost
        5. Apply cost-sensitive threshold
        6. Display prediction
        """
    )

    st.markdown("---")

    st.success("🟢 Model Ready")


# =========================================================
# INTRODUCTION CARD
# =========================================================

st.markdown(
    """
    <div class="card">

    <div class="card-title">
    🔍 APS Failure Detection
    </div>

    <div class="card-text">
    Upload Scania truck operational data and let the trained
    machine-learning model identify trucks that may belong to
    the APS-failure class.
    </div>

    </div>
    """,
    unsafe_allow_html=True
)


# =========================================================
# MODE SELECTION
# =========================================================

st.markdown("### Select Analysis Mode")

mode = st.radio(
    "",
    [
        "🚛 Predict New Trucks",
        "📊 Evaluate Labeled Data"
    ],
    horizontal=True
)


# =========================================================
# FILE UPLOAD
# =========================================================

st.markdown("### Upload Truck Data")

uploaded_file = st.file_uploader(
    "Upload CSV file",
    type=["csv"],
    help=(
        "Upload a Scania CSV containing the required "
        "operational features."
    )
)


# =========================================================
# PREDICTION
# =========================================================

if uploaded_file is not None:

    st.success(
        f"✅ File uploaded: {uploaded_file.name}"
    )

    st.markdown(
        '<div class="custom-divider"></div>',
        unsafe_allow_html=True
    )

    if st.button("🚀 Run ScaniaGuard Analysis"):

        try:

            # -------------------------------------------------
            # Run model
            # -------------------------------------------------

            predictions, probabilities, y_true = opener(
                uploaded_file
            )

            total_trucks = len(predictions)

            predicted_failures = int(
                (predictions == 1).sum()
            )

            predicted_normal = int(
                (predictions == 0).sum()
            )


            # =================================================
            # SUMMARY
            # =================================================

            st.markdown("## 📊 Analysis Summary")

            col1, col2, col3 = st.columns(3)

            with col1:

                st.markdown(
                    f"""
                    <div class="status-card">

                    <div class="status-number">
                    {total_trucks:,}
                    </div>

                    <div class="status-label">
                    Total Trucks
                    </div>

                    </div>
                    """,
                    unsafe_allow_html=True
                )

            with col2:

                st.markdown(
                    f"""
                    <div class="status-card">

                    <div class="status-number">
                    {predicted_failures:,}
                    </div>

                    <div class="status-label">
                    APS Failures Detected
                    </div>

                    </div>
                    """,
                    unsafe_allow_html=True
                )

            with col3:

                st.markdown(
                    f"""
                    <div class="status-card">

                    <div class="status-number">
                    {predicted_normal:,}
                    </div>

                    <div class="status-label">
                    Not APS Failure
                    </div>

                    </div>
                    """,
                    unsafe_allow_html=True
                )


            # =================================================
            # EVALUATION MODE
            # =================================================

            if mode == "📊 Evaluate Labeled Data":

                if y_true is None:

                    st.warning(
                        "⚠️ This CSV does not contain a "
                        "`class` column. Evaluation requires "
                        "actual labels."
                    )

                else:

                    st.markdown("## 📈 Model Evaluation")

                    tp = int(
                        (
                            (y_true == 1) &
                            (predictions == 1)
                        ).sum()
                    )

                    tn = int(
                        (
                            (y_true == 0) &
                            (predictions == 0)
                        ).sum()
                    )

                    fp = int(
                        (
                            (y_true == 0) &
                            (predictions == 1)
                        ).sum()
                    )

                    fn = int(
                        (
                            (y_true == 1) &
                            (predictions == 0)
                        ).sum()
                    )


                    # Cost
                    total_cost = (
                        fp * 10 +
                        fn * 500
                    )


                    # Metrics
                    recall = (
                        tp / (tp + fn)
                        if (tp + fn) > 0
                        else 0
                    )

                    precision = (
                        tp / (tp + fp)
                        if (tp + fp) > 0
                        else 0
                    )


                    c1, c2, c3, c4 = st.columns(4)

                    c1.metric(
                        "True Positives",
                        tp
                    )

                    c2.metric(
                        "False Positives",
                        fp
                    )

                    c3.metric(
                        "False Negatives",
                        fn
                    )

                    c4.metric(
                        "True Negatives",
                        tn
                    )


                    st.markdown("### 💰 Cost Analysis")

                    cost_col1, cost_col2 = st.columns(2)

                    with cost_col1:

                        st.metric(
                            "Total Cost",
                            f"₹{total_cost:,}"
                        )

                    with cost_col2:

                        st.metric(
                            "False Negative Cost",
                            f"₹{fn * 500:,}"
                        )


                    metric_col1, metric_col2 = st.columns(2)

                    metric_col1.metric(
                        "Recall",
                        f"{recall:.2%}"
                    )

                    metric_col2.metric(
                        "Precision",
                        f"{precision:.2%}"
                    )


            # =================================================
            # NEW TRUCK PREDICTION MODE
            # =================================================

            else:

                st.markdown("## 🚛 Prediction Results")

                if predicted_failures > 0:

                    st.markdown(
                        f"""
                        <div class="risk-high">

                        <div class="risk-title">
                        🚨 APS Failure Risk Detected
                        </div>

                        <div class="card-text">
                        {predicted_failures:,} truck(s)
                        crossed the cost-sensitive
                        decision threshold.
                        </div>

                        </div>
                        """,
                        unsafe_allow_html=True
                    )

                else:

                    st.markdown(
                        """
                        <div class="risk-low">

                        <div class="risk-title">
                        ✅ No APS Failure Detected
                        </div>

                        <div class="card-text">
                        No uploaded truck crossed the
                        decision threshold.
                        </div>

                        </div>
                        """,
                        unsafe_allow_html=True
                    )


            # =================================================
            # RESULTS TABLE
            # =================================================

            st.markdown("## 🔎 Detailed Predictions")

            result_df = pd.DataFrame({
                "Truck": range(
                    1,
                    total_trucks + 1
                ),

                "Failure Probability": probabilities,

                "Prediction": predictions
            })


            result_df["Prediction"] = (
                result_df["Prediction"]
                .map({
                    0: "🟢 Not APS Failure",
                    1: "🔴 APS Failure"
                })
            )


            result_df["Failure Probability"] = (
                result_df["Failure Probability"]
                .map(
                    lambda x: f"{x:.2%}"
                )
            )


            st.dataframe(
                result_df,
                use_container_width=True,
                hide_index=True
            )


            # =================================================
            # DOWNLOAD RESULTS
            # =================================================

            download_df = pd.DataFrame({
                "Truck": range(
                    1,
                    total_trucks + 1
                ),

                "Failure_Probability": probabilities,

                "Prediction": predictions
            })


            csv = download_df.to_csv(
                index=False
            ).encode("utf-8")


            st.download_button(
                label="⬇️ Download Predictions",
                data=csv,
                file_name="scaniaguard_predictions.csv",
                mime="text/csv"
            )


        except Exception as e:

            st.error(
                f"Prediction failed: {e}"
            )

In [ ]:
import streamlit as st
import pandas as pd
import base64

from helper import opener

def html(content):
    st.markdown(
        content,
        unsafe_allow_html=True
    )
# =========================================================
# PAGE CONFIGURATION
# =========================================================

st.set_page_config(
    page_title="ScaniaGuard",
    page_icon="🚛",
    layout="wide",
    initial_sidebar_state="expanded"
)


# =========================================================
# BACKGROUND IMAGE
# =========================================================

def set_background(image_path):

    with open(image_path, "rb") as image_file:

        encoded_image = base64.b64encode(
            image_file.read()
        ).decode()

    st.markdown(
        f"""
        <style>

        .stApp {{
            background-image:
                linear-gradient(
                    rgba(5, 10, 20, 0.88),
                    rgba(5, 10, 20, 0.94)
                ),
                url("data:image/jpg;base64,{encoded_image}");

            background-size: cover;
            background-position: center;
            background-attachment: fixed;
        }}

        </style>
        """,
        unsafe_allow_html=True
    )


set_background("assets/scania_truck.jpg")


# =========================================================
# CUSTOM CSS
# =========================================================

st.markdown(
    """
    <style>

    /* Main container */

    .block-container {
        padding-top: 1.5rem;
        padding-bottom: 3rem;
        max-width: 1400px;
    }


    /* Sidebar */

    section[data-testid="stSidebar"] {
        background: rgba(7, 13, 25, 0.96);
    }


    /* Main title */

    .hero-title {
        text-align: center;
        font-size: 4rem;
        font-weight: 900;
        letter-spacing: 3px;
        margin-top: 15px;
        margin-bottom: 5px;
    }


    .hero-subtitle {
        text-align: center;
        font-size: 1.35rem;
        color: #b8c2d1;
        margin-bottom: 8px;
    }


    .hero-description {
        text-align: center;
        font-size: 0.95rem;
        color: #8491a5;
        margin-bottom: 30px;
    }


    /* Glass cards */

    .glass-card {
        background: rgba(15, 23, 42, 0.72);
        border: 1px solid rgba(255, 255, 255, 0.10);
        border-radius: 18px;
        padding: 25px;
        margin-bottom: 20px;
        backdrop-filter: blur(12px);
    }


    .card-title {
        font-size: 1.25rem;
        font-weight: 750;
        margin-bottom: 8px;
    }


    .card-description {
        color: #aab5c5;
        font-size: 0.95rem;
    }


    /* Model status cards */

    .status-card {
        background: rgba(15, 23, 42, 0.78);
        border: 1px solid rgba(255, 255, 255, 0.10);
        border-radius: 14px;
        padding: 15px 10px;
        text-align: center;
        min-height: 75px;
    }


    .status-value {
        font-size: 1.15rem;
        font-weight: 750;
    }


    .status-label {
        color: #8995a8;
        font-size: 0.78rem;
        margin-top: 3px;
    }


    /* Result cards */

    .result-card {
        background: rgba(15, 23, 42, 0.80);
        border: 1px solid rgba(255, 255, 255, 0.10);
        border-radius: 16px;
        padding: 20px;
        text-align: center;
    }


    .result-number {
        font-size: 2rem;
        font-weight: 800;
    }


    .result-label {
        color: #9ba7b8;
        font-size: 0.85rem;
    }


    /* Failure result */

    .failure-card {
        background: rgba(185, 28, 28, 0.18);
        border: 1px solid rgba(239, 68, 68, 0.45);
        border-radius: 18px;
        padding: 28px;
        text-align: center;
        margin: 20px 0;
    }


    .failure-title {
        font-size: 1.8rem;
        font-weight: 850;
    }


    .success-card {
        background: rgba(21, 128, 61, 0.18);
        border: 1px solid rgba(34, 197, 94, 0.40);
        border-radius: 18px;
        padding: 28px;
        text-align: center;
        margin: 20px 0;
    }


    .success-title {
        font-size: 1.8rem;
        font-weight: 850;
    }


    /* Cost card */

    .cost-card {
        background: rgba(217, 119, 6, 0.15);
        border: 1px solid rgba(245, 158, 11, 0.40);
        border-radius: 18px;
        padding: 25px;
        text-align: center;
        margin: 15px 0 25px 0;
    }


    .cost-number {
        font-size: 2.4rem;
        font-weight: 850;
    }


    .cost-label {
        color: #c3cad4;
        margin-top: 5px;
    }


    /* Divider */

    .divider {
        height: 1px;
        background: rgba(255,255,255,0.10);
        margin: 28px 0;
    }


    /* Buttons */

    .stButton > button {
        width: 100%;
        height: 3.2rem;
        border-radius: 12px;
        font-size: 1rem;
        font-weight: 750;
    }


    /* File uploader */

    [data-testid="stFileUploader"] {
        background: rgba(15, 23, 42, 0.55);
        border-radius: 15px;
        padding: 8px;
    }


    </style>
    """,
    unsafe_allow_html=True
)


# =========================================================
# HERO SECTION
# =========================================================

st.markdown(
    """
    <div class="hero-title">
        🚛 SCANIAGUARD
    </div>

    <div class="hero-subtitle">
        Cost-Sensitive APS Failure Detection
    </div>

    <div class="hero-description">
        Machine Learning System for Predictive Truck Failure Detection
    </div>
    """,
    unsafe_allow_html=True
)


# =========================================================
# MODEL STATUS
# =========================================================

status1, status2, status3, status4 = st.columns(4)


with status1:

    html(
        """
        <div class="status-card">

            <div class="status-value">
                🟢 ONLINE
            </div>

            <div class="status-label">
                Model Status
            </div>

        </div>
        """,
       
    )


with status2:

    html(
        """
        <div class="status-card">

            <div class="status-value">
                XGBoost
            </div>

            <div class="status-label">
                ML Model
            </div>

        </div>
        """,
       
    )


with status3:

    html(
        """
        <div class="status-card">

            <div class="status-value">
                236
            </div>

            <div class="status-label">
                Features
            </div>

        </div>
        """,
        
    )


with status4:

    html(
        """
        <div class="status-card">

            <div class="status-value">
                0.024
            </div>

            <div class="status-label">
                Decision Threshold
            </div>

        </div>
        """,
        
    )


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.markdown("## 🚛 ScaniaGuard")

    st.markdown("### How It Works")

    st.write(
        """
        **1.** Upload truck data

        **2.** Handle missing values

        **3.** Generate missingness indicators

        **4.** Run XGBoost

        **5.** Apply cost-sensitive threshold

        **6.** Display prediction
        """
    )

    st.markdown("---")

    


# =========================================================
# INTRODUCTION
# =========================================================

html(
    """
    <div class="glass-card">

        <div class="card-title">
            🔍 APS Failure Detection
        </div>

        <div class="card-description">
            Upload Scania truck operational data and use the trained
            machine-learning model to identify trucks that may belong
            to the APS-failure class.
        </div>

    </div>
    """,

)


# =========================================================
# MODE SELECTION
# =========================================================

st.markdown("### Select Analysis Mode")

mode = st.radio(
    "",
    [
        "🚛 Predict New Trucks",
        "📊 Evaluate Labeled Data"
    ],
    horizontal=True
)


# =========================================================
# FILE UPLOAD
# =========================================================

st.markdown("### Upload Truck Data")

uploaded_file = st.file_uploader(
    "Upload CSV file",
    type=["csv"],
    help=(
        "Upload a Scania CSV containing the required "
        "operational features."
    )
)


# =========================================================
# RUN ANALYSIS
# =========================================================

if uploaded_file is not None:

    st.success(
        f"✅ File uploaded: {uploaded_file.name}"
    )

    html(
        '<div class="divider"></div>',
    )

    if st.button("🚀 Run ScaniaGuard Analysis"):

        try:

            # -------------------------------------------------
            # RUN MODEL
            # -------------------------------------------------

            predictions, probabilities, y_true = opener(
                uploaded_file
            )

            total_trucks = len(predictions)

            predicted_failures = int(
                (predictions == 1).sum()
            )

            predicted_normal = int(
                (predictions == 0).sum()
            )


            # =================================================
            # SUMMARY
            # =================================================

            st.markdown("## 📊 Analysis Summary")

            col1, col2, col3 = st.columns(3)


            with col1:

                html(
                    f"""
                    <div class="result-card">

                        <div class="result-number">
                            {total_trucks:,}
                        </div>

                        <div class="result-label">
                            Total Trucks
                        </div>

                    </div>
                    """,
                   
                )


            with col2:

                html(
                    f"""
                    <div class="result-card">

                        <div class="result-number">
                            {predicted_failures:,}
                        </div>

                        <div class="result-label">
                            APS Failures Detected
                        </div>

                    </div>
                    """,
                    
                )


            with col3:

                html(
                    f"""
                    <div class="result-card">

                        <div class="result-number">
                            {predicted_normal:,}
                        </div>

                        <div class="result-label">
                            Not APS Failure
                        </div>

                    </div>
                    """,
                   
                )


            # =================================================
            # EVALUATION MODE
            # =================================================

            if mode == "📊 Evaluate Labeled Data":

                if y_true is None:

                    st.warning(
                        "⚠️ This CSV does not contain a "
                        "`class` column. Evaluation requires "
                        "actual labels."
                    )

                else:

                    st.markdown(
                        "## 📈 Model Evaluation"
                    )

                    tp = int(
                        (
                            (y_true == 1) &
                            (predictions == 1)
                        ).sum()
                    )

                    tn = int(
                        (
                            (y_true == 0) &
                            (predictions == 0)
                        ).sum()
                    )

                    fp = int(
                        (
                            (y_true == 0) &
                            (predictions == 1)
                        ).sum()
                    )

                    fn = int(
                        (
                            (y_true == 1) &
                            (predictions == 0)
                        ).sum()
                    )


                    # -----------------------------------------
                    # COST
                    # -----------------------------------------

                    fp_cost = fp * 10
                    fn_cost = fn * 500

                    total_cost = (
                        fp_cost +
                        fn_cost
                    )


                    # -----------------------------------------
                    # METRICS
                    # -----------------------------------------

                    recall = (
                        tp / (tp + fn)
                        if (tp + fn) > 0
                        else 0
                    )

                    precision = (
                        tp / (tp + fp)
                        if (tp + fp) > 0
                        else 0
                    )


                    # -----------------------------------------
                    # CONFUSION MATRIX COUNTS
                    # -----------------------------------------

                    c1, c2, c3, c4 = st.columns(4)


                    c1.metric(
                        "True Positives",
                        tp
                    )

                    c2.metric(
                        "False Positives",
                        fp
                    )

                    c3.metric(
                        "False Negatives",
                        fn
                    )

                    c4.metric(
                        "True Negatives",
                        tn
                    )


                    # -----------------------------------------
                    # COST DISPLAY
                    # -----------------------------------------

                    html(
                        f"""
                        <div class="cost-card">

                            <div class="cost-number">
                                ₹{total_cost:,}
                            </div>

                            <div class="cost-label">
                                Total Misclassification Cost
                            </div>

                        </div>
                        """,
                        
                    )


                    cost1, cost2 = st.columns(2)


                    with cost1:

                        st.metric(
                            "False Positive Cost",
                            f"₹{fp_cost:,}"
                        )


                    with cost2:

                        st.metric(
                            "False Negative Cost",
                            f"₹{fn_cost:,}"
                        )


                    # -----------------------------------------
                    # PERFORMANCE
                    # -----------------------------------------

                    metric1, metric2 = st.columns(2)


                    with metric1:

                        st.metric(
                            "Recall",
                            f"{recall:.2%}"
                        )


                    with metric2:

                        st.metric(
                            "Precision",
                            f"{precision:.2%}"
                        )


            # =================================================
            # PREDICTION MODE
            # =================================================

            else:

                st.markdown(
                    "## 🚛 Prediction Results"
                )


                if predicted_failures > 0:

                    html(
                        f"""
                        <div class="failure-card">

                            <div class="failure-title">
                                🚨 APS Failure Risk Detected
                            </div>

                            <div class="card-description">
                                {predicted_failures:,}
                                truck(s) crossed the
                                cost-sensitive decision
                                threshold.
                            </div>

                        </div>
                        """,
                        
                    )

                else:

                    html(
                        """
                        <div class="success-card">

                            <div class="success-title">
                                ✅ No APS Failure Detected
                            </div>

                            <div class="card-description">
                                No uploaded truck crossed
                                the decision threshold.
                            </div>

                        </div>
                        """,
                       
                    )


            # =================================================
            # RESULTS TABLE
            # =================================================

            st.markdown(
                "## 🔎 Detailed Predictions"
            )


            result_df = pd.DataFrame({

                "Truck": range(
                    1,
                    total_trucks + 1
                ),

                "Failure Probability": probabilities,

                "Prediction": predictions

            })


            result_df["Prediction"] = (
                result_df["Prediction"]
                .map({
                    0: "🟢 Not APS Failure",
                    1: "🔴 APS Failure"
                })
            )


            result_df["Failure Probability"] = (
                result_df[
                    "Failure Probability"
                ]
                .map(
                    lambda x: f"{x:.2%}"
                )
            )


            st.dataframe(
                result_df,
                use_container_width=True,
                hide_index=True
            )


            # =================================================
            # DOWNLOAD RESULTS
            # =================================================

            download_df = pd.DataFrame({

                "Truck": range(
                    1,
                    total_trucks + 1
                ),

                "Failure_Probability": probabilities,

                "Prediction": predictions

            })


            csv = download_df.to_csv(
                index=False
            ).encode("utf-8")


            st.download_button(
                label="⬇️ Download Predictions",
                data=csv,
                file_name="scaniaguard_predictions.csv",
                mime="text/csv"
            )


        except Exception as e:

            st.error(
                f"Prediction failed: {e}"
            )

